In [ ]:
!pip install -q "transformers==4.43.3" "datasets" "accelerate" "sentencepiece" "protobuf==3.20.3" scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-dataplex 2.18.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.26.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
wandb 0.25.1 requires protobuf!=5.28.0,!=5.29.0,<7,>4.21.0, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.1 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 3.20.3 w

In [4]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from sklearn.metrics import classification_report, confusion_matrix

In [8]:
dataset = load_dataset("ag_news")
train_subset = dataset['train'].shuffle(seed=42).select(range(10000))
test_subset = dataset['test'].shuffle(seed=42).select(range(2000))

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [9]:
labels_names = dataset['train'].features['label'].names
id2label = {i: name for i, name in enumerate(labels_names)}
label2id = {name: i for i, name in enumerate(labels_names)}

model_id = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [ ]:
def preprocess_function(examples):
    result = tokenizer(examples["text"], truncation=True, padding='max_length', max_length=128)
    result["labels"] = examples["label"]
    return result

In [ ]:
tokenized_train = train_subset.map(preprocess_function, batched=True, remove_columns=dataset['train'].column_names)
tokenized_test = test_subset.map(preprocess_function, batched=True, remove_columns=dataset['test'].column_names)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=len(labels_names), id2label=id2label, label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args = TrainingArguments(
    output_dir="pro_news_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False,
    max_grad_norm=1.0,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.483700,0.323232
2,0.279200,0.327721
3,0.241000,0.326961


TrainOutput(global_step=1875, training_loss=0.30588267415364584, metrics={'train_runtime': 593.0221, 'train_samples_per_second': 50.588, 'train_steps_per_second': 3.162, 'total_flos': 993576314880000.0, 'train_loss': 0.30588267415364584, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("\n--- Classification Report ---")
print(classification_report(true_labels, preds, target_names=labels_names))


--- Classification Report ---
              precision    recall  f1-score   support

       World       0.92      0.88      0.90       497
      Sports       0.94      0.99      0.97       483
    Business       0.86      0.85      0.86       522
    Sci/Tech       0.86      0.87      0.87       498

    accuracy                           0.90      2000
   macro avg       0.90      0.90      0.90      2000
weighted avg       0.90      0.90      0.90      2000



In [7]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
trainer.push_to_hub("immrass/news-classifier-deberta")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ssifier/model.safetensors:   0%|          | 1.15MB /  568MB            

  ...ssifier/training_args.bin:  28%|##8       | 1.58kB / 5.58kB            

CommitInfo(commit_url='https://huggingface.co/immrass/pro_news_classifier/commit/2e1ffcbfab4c37ac05ce74b9eeb1589692527993', commit_message='immrass/news-classifier-deberta', commit_description='', oid='2e1ffcbfab4c37ac05ce74b9eeb1589692527993', pr_url=None, repo_url=RepoUrl('https://huggingface.co/immrass/pro_news_classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='immrass/pro_news_classifier'), pr_revision=None, pr_num=None)

In [10]:
tokenizer.push_to_hub("immrass/pro_news_classifier")

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/immrass/pro_news_classifier/commit/e2c1efbc745aebbddfd296da97529f8a3e1eb508', commit_message='Upload tokenizer', commit_description='', oid='e2c1efbc745aebbddfd296da97529f8a3e1eb508', pr_url=None, repo_url=RepoUrl('https://huggingface.co/immrass/pro_news_classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='immrass/pro_news_classifier'), pr_revision=None, pr_num=None)

In [13]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

model_id = "immrass/pro_news_classifier"

model = AutoModelForSequenceClassification.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

In [16]:
test_cases = [
    # 1. Sci/Tech
    "SpaceX successfully landed its Starship rocket after a high-altitude flight test.",
    "Scientists have developed a new quantum processor that outperforms classical supercomputers.",
    "The new software update fixes a critical security vulnerability in the operating system.",

    # 2. Business
    "Central banks are considering raising interest rates to combat rising inflation.",
    "Microsoft shares rose after the company announced its fiscal results and dividend payments.",
    "Oil prices dropped significantly today due to concerns about global demand.",

    # 3. Sports
    "The Golden State Warriors secured a spot in the NBA playoffs after a thrilling victory.",
    "Novak Djokovic wins his 24th Grand Slam title at the US Open final.",
    "The upcoming World Cup will feature teams from 48 different nations for the first time.",

    # 4. World
    "Leaders of the G7 nations met in London to discuss climate change and global security.",
    "A massive earthquake has struck the coast, triggering emergency rescue operations.",
    "Diplomatic talks have begun between the two countries to resolve the long-standing border dispute."
]

print("-" * 30)
for text in test_cases:
    res = classifier(text)[0]
    print(f"Text: {text[:60]}...")
    print(f"Result: {res['label']} ({res['score']:.4f})")
    print("-" * 30)

------------------------------
Text: SpaceX successfully landed its Starship rocket after a high-...
Result: Sci/Tech (0.8639)
------------------------------
Text: Scientists have developed a new quantum processor that outpe...
Result: Sci/Tech (0.8757)
------------------------------
Text: The new software update fixes a critical security vulnerabil...
Result: Sci/Tech (0.8898)
------------------------------
Text: Central banks are considering raising interest rates to comb...
Result: Business (0.9408)
------------------------------
Text: Microsoft shares rose after the company announced its fiscal...
Result: Business (0.9539)
------------------------------
Text: Oil prices dropped significantly today due to concerns about...
Result: Business (0.9411)
------------------------------
Text: The Golden State Warriors secured a spot in the NBA playoffs...
Result: Sports (0.7595)
------------------------------
Text: Novak Djokovic wins his 24th Grand Slam title at the US Open...
Result: Spor